#### Libraries

In [ ]:
import pandas as pd
import os
import re
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib.dates as mdates
import numpy as np
import matplotlib.patches as mpatches

#### Outlet-Level Thermal Analysis

**Goal:** Build an outlet-level, day-granular timeline across CTD, CLG, PLE, and Session logs to study thermal anomalies and PLE/ple spikes.  

#### 1) Dataset schema audit
**Goal:** Inspect columns, dtypes and a few rows.  

In [ ]:
# 📁 Define paths
DATA_DIR = os.path.join("..", "data_all")

In [ ]:
ctd_df = pd.read_csv(os.path.join(DATA_DIR, "CTD_last_year.csv"))
# clg_df = pd.read_csv(os.path.join(DATA_DIR, "CLG_total.xlsx"))
ple_df = pd.read_csv(os.path.join(DATA_DIR, "PLE_last_year.csv"))
sess_df = pd.read_csv(os.path.join(DATA_DIR, "SeccSessionStop_last_year.csv"))


# 🔍 Function to summarize
def summarize_dataset(name, df):
    print(f"\n=== 🔎 {name} Dataset ===")
    print("🧾 Columns:", df.columns.tolist())
    print("📊 Dtypes:")
    print(df.dtypes)
    print("👀 Head:")
    display(df.head(5))

# 📥 Load and summarize each
for name, df in [
    ("CTD", ctd_df),
    # ("CLG", clg_df),
    ("PLE", ple_df),
    ("SeccSessionStop", sess_df)
]:
    summarize_dataset(name, df)


#### 2) PLE: Extract outlet from @message
**Purpose:** Parse outlet id (DC1/DC2/…) from PLE logs.  

In [ ]:
# # 📁 Load dataset
# DATA_DIR = os.path.join("..", "data_all")
# ple_path = os.path.join(DATA_DIR, "PLE.csv")
# ple_df = pd.read_csv(ple_path)

# # ✅ Step 1: Parse outlet info from `@message`
# ple_df["outlet"] = ple_df["@ptr"].str.extract(r"(DC\d+)_CableTempSensor")

# # 🔍 Optional check: Print unique outlets
# print("✅ Extracted outlets:", ple_df["outlet"].dropna().unique())

# # ✅ Step 2: Save the updated file back
# ple_df.to_csv(ple_path, index=False)
# print("✅ PLE_timeseries.xlsx updated with 'outlet' column.")


- 3) Normalize CTD outlets
**Purpose:** Extract integer outlet id from `cableid`.  
- 4) Normalize CLG outlets
**Purpose:** Extract integer outlet id from `cableid`.
- 5) Normalize PLE outlets
**Purpose:** Ensure integer `outlet` in PLE.  
- 6) Normalize Session outlets
**Purpose:** Strip quotes and parse `outlet` to integer.  

In [ ]:
# === Step 1: Split IDOutlet into @logStream and outlet ===

def split_idoutlet(id_str):
    """
    Split IDOutlet into charger (@logStream) and outlet (last digit).
    """
    if pd.isna(id_str):
        return (pd.NA, pd.NA)
    s = str(id_str)
    if s[-1].isdigit():
        return (s[:-1], int(s[-1]))
    else:
        return (s, pd.NA)

# --- Apply to CTD dataset ---
ctd_split = ctd_df["IDOutlet"].apply(split_idoutlet)
ctd_df["@logStream"] = ctd_split.apply(lambda x: x[0])
ctd_df["outlet"] = ctd_split.apply(lambda x: x[1]).astype("Int64")

# --- Apply to PLE dataset ---
ple_split = ple_df["IDOutlet"].apply(split_idoutlet)
ple_df["@logStream"] = ple_split.apply(lambda x: x[0])
ple_df["outlet"] = ple_split.apply(lambda x: x[1]).astype("Int64")


# Verify cableid consistency (last digit-CTD)
if "cableid" in ctd_df.columns:
    ctd_df["cableid_outlet"] = ctd_df["cableid"].apply(lambda x: int(str(x)[-1]) if pd.notna(x) and str(x)[-1].isdigit() else pd.NA).astype("Int64")
    mismatches = ctd_df[ctd_df["outlet"] != ctd_df["cableid_outlet"]]
    if not mismatches.empty:
        print("⚠️ Mismatches found between IDOutlet and cableid:")
        display(mismatches.head(10))
    else:
        print("✅ All CTD rows consistent: IDOutlet and cableid match.")

# --- Apply to SeccSessionStop dataset ---
sess_split = sess_df["IDOutlet"].apply(split_idoutlet)
sess_df["@logStream"] = sess_split.apply(lambda x: x[0])
sess_df["outlet"] = sess_split.apply(lambda x: x[1]).astype("Int64")

print("✅ @logStream + outlet columns created for CTD and SeccSessionStop.")
print("CTD sample:", ctd_df[["@logStream", "outlet"]].head())
print("PLE sample:", ple_df[["@logStream", "outlet"]].head())
print("Sessions sample:", sess_df[["@logStream", "outlet"]].head())


In [ ]:
for df in [ctd_df, sess_df, ple_df]:
    df["@logStream"] = df["@logStream"].astype(str).str.strip()


#### 7) Add day granularity
- All datasets now have a `day` column. Day-level aggregation reduces noise and aligns signals

In [ ]:
#7 Add day column
for df in [ctd_df, sess_df, ple_df]:
    df["@timestamp"] = pd.to_datetime(df["@timestamp"], errors='coerce')
    df["day"] = df["@timestamp"].dt.floor("D")

#### 8) Safety: sort timestamps
**Purpose:** Sort by `@timestamp` before merge. 

In [ ]:
# After loading and timestamp conversion:
for df in [ctd_df, sess_df, ple_df]:
    df["@timestamp"] = pd.to_datetime(df["@timestamp"], errors='coerce')
    df.sort_values(by="@timestamp", inplace=True)
    df["day"] = df["@timestamp"].dt.floor("D")

- 9) CTD aggregation: Purpose:** Aggregate CTD by outlet-day.  
- 10) CLG aggregation
- 11) PLE aggregation
- 12) Session aggregation

In [ ]:
#9
ctd_summary = ctd_df.groupby(["@logStream", "outlet", "day"]).agg(
    CTD_count=("@timestamp", "count"),
    CTD_diff_mean=("diff", "mean"),
    CTD_factor_mean=("factor", "mean"),
    CTD_current_mean=("nowcur", "mean")
).reset_index()

In [ ]:
#11
ple_summary = ple_df.groupby(["@logStream", "outlet", "day"]).agg(
    PLE_count=("@timestamp", "count")
).reset_index()

In [ ]:
#12 Re-aggregate Sessions with New features(Duration)
sess_summary = sess_df.groupby(["@logStream", "outlet", "day"]).agg(
    Sess_count=("@timestamp", "count"),
    Sess_energy_mean=("energy", "mean"),
    Sess_temp_diff_mean=("diff", "mean"),
    # NEW:
    Sess_duration_mean=("duration", "mean"),   # avg duration per day (e.g., seconds or minutes)
    Sess_duration_total=("duration", "sum")    # total duration per day
).reset_index()

#### 13) Build master logstream-outlet-timeline
**Purpose:** Merge CTD, CLG, PLE, Sessions on logstream-outlet-day.  

In [ ]:
master_timeline = (
    ctd_summary.merge(sess_summary, on=["@logStream", "outlet", "day"], how="outer")
               .merge(ple_summary, on=["@logStream", "outlet", "day"], how="outer")
)

# Preview
print("✅ Outlet-normalized timeline built!")
print(master_timeline.shape)
display(master_timeline.head())

- clean_master_timeline

- Counts (*_count) → fill missing with 0 (no event that day).

- Means (*_mean) → keep NaN (no measurement; don’t invent zeros).

- Make sure numeric columns are really numeric.

In [ ]:
def clean_master_timeline(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # --- 1) Identify feature columns by suffix
    count_cols = [c for c in df.columns if c.endswith("_count")]
    mean_cols  = [c for c in df.columns if c.endswith("_mean")]

    # --- 2) Force numeric on feature columns (bad strings -> NaN)
    for c in count_cols + mean_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # --- 3) Fill counts with 0 (no event on that day)
    df[count_cols] = df[count_cols].fillna(0).astype("float64")

    # --- 4) Mask *means* by their family’s count (0 or NaN -> mean must be NaN)
    families = {
        "CTD": ["CTD_diff_mean", "CTD_factor_mean", "CTD_current_mean"],
        # "CLG": ["CLG_diff_mean", "CLG_temp1_mean", "CLG_temp2_mean", "CLG_current_mean"],
        # PLE has only counts
    }

    for fam, cols in families.items():
        cnt = f"{fam}_count"
        if cnt in df.columns:
            mask = (df[cnt].isna()) | (df[cnt] == 0)
            for col in cols:
                if col in df.columns:
                    # Any day with count==0/NaN cannot have a valid mean -> set to NaN
                    df.loc[mask, col] = np.nan

    # --- 5) Ensure key columns typed correctly
    if "@timestamp" in df.columns:
        df["@timestamp"] = pd.to_datetime(df["@timestamp"], errors="coerce")
    if "day" in df.columns:
        df["day"] = pd.to_datetime(df["day"], errors="coerce")
    if "outlet" in df.columns:
        df["outlet"] = pd.to_numeric(df["outlet"], errors="coerce").astype("Int64")

    return df

# 👉 apply right after the outer merges
master_timeline = clean_master_timeline(master_timeline)

print("✅ Cleaned + masked timeline.")
print("Counts (zero-filled):", [c for c in master_timeline.columns if c.endswith("_count")])
print("Means (masked by counts):", [c for c in master_timeline.columns if c.endswith("_mean")])

- convert ms -> minutes BEFORE grouping


In [ ]:
#convert ms -> minutes BEFORE grouping
sess_df["duration"] = pd.to_numeric(sess_df["duration"], errors="coerce")
sess_df["duration_min"] = sess_df["duration"] / 60000.0  # ms to minutes
# then aggregate on "duration_min" instead of "duration"

In [ ]:
print(master_timeline.shape)
display(master_timeline.head(400))

#### 14) Completeness check
**Purpose:** Count NaNs per feature.  

In [ ]:
# 📝 List of columns to check
columns_to_check = [
   "CTD_count","Sess_count", "PLE_count"
]

# 🔍 Count zeros in each
zero_counts = (master_timeline[columns_to_check] == 0).sum()

# 🔢 Also show total number of rows for reference
total_rows = len(master_timeline)

# 🖨️ Print report
print(f"Total rows: {total_rows}\n")
for col in columns_to_check:
    count = int(zero_counts[col])
    pct = round(100 * count / total_rows, 2) if total_rows else 0
    if count == total_rows:
        print(f"⚠️ Column '{col}' has ALL {count} zeros ❗")
    else:
        print(f"✅ Column '{col}' has {count} zeros ({pct}%)")


- Why did Sess_count become 0?
- In the outer merge, if on a given day + charger + outlet there were no sessions recorded, the merged row is created anyway (due to CTD/CLG/PLE presence, or just the calendar alignment).

### Filter the Outlets based on slelected chargers!

In [ ]:
#Sweden
# selected_chargers = [
#     "LCk4jo", "gwmF5L", "UzOYLO","GHTN2X","THLN7m","JlBtzX","eZCh8O"
# ]

#Finland
selected_chargers = [
    "Hy6NfZ","YX5xq0","BkJsud","nFJBZS","ZIJhLf","AZNNJA","z4gi5Z","acno33",
    "GcIKxl","udABVw","hE5gvw","Mgx7DR","ZligFW","N3Btwu","sJT41a","lpuy1j",
    "liZll0","hJYSAt","fXnldw","nqq1M7","XzC5nW","jKNdUi","V2KgdI","56VYaM",
    "LIjq41","WJv88t","rI01JZ","kXnNip","Mu3puO","PdbewO","gkybxl","2H1Hul"
]


for name, df in [("CTD", ctd_df), ("Sess", sess_df), ("PLE", ple_df)]:
    print(f"\n{name} availability:")
    avail = df[df["@logStream"].isin(selected_chargers)].groupby("@logStream")["outlet"].nunique()
    print(avail)


##### Filtering + Aggregation + Master Timeline (correct order)


In [ ]:
common_chargers = (
    set(ctd_df["@logStream"].unique())
    & set(sess_df["@logStream"].unique())
    & set(ple_df["@logStream"].unique())
)

filtered_chargers = [c for c in selected_chargers if c in common_chargers]
print("✅ Chargers with all three datasets:", len(filtered_chargers), filtered_chargers)


In [ ]:
ctd_filtered = ctd_df[ctd_df["@logStream"].isin(selected_chargers)].copy()
sess_filtered = sess_df[sess_df["@logStream"].isin(selected_chargers)].copy()
ple_filtered = ple_df[ple_df["@logStream"].isin(selected_chargers)].copy()

In [ ]:
print("✅ Filtering complete.")
print("CTD rows before:", len(ctd_df), "→ after:", len(ctd_filtered))
print("Sessions rows before:", len(sess_df), "→ after:", len(sess_filtered))
print("PLE rows before:", len(ple_df), "→ after:", len(ple_filtered))

print("Unique outlets (CTD):", ctd_filtered["outlet"].nunique())
print("Unique outlets (Sessions):", sess_filtered["outlet"].nunique())
print("Unique outlets (PLE):", ple_filtered["outlet"].nunique())

In [ ]:
for name, df in [("CTD", ctd_df), ("Sess", sess_df), ("PLE", ple_df)]:
    print(f"{name}:")
    print(df.groupby("@logStream")["outlet"].nunique())


- Aggregations

In [ ]:
ctd_summary = ctd_filtered.groupby(["@logStream", "outlet", "day"]).agg(
    CTD_count=("@timestamp", "count"),
    CTD_diff_mean=("diff", "mean"),
    CTD_factor_mean=("factor", "mean"),
    CTD_current_mean=("nowcur", "mean")
).reset_index()

sess_summary = sess_filtered.groupby(["@logStream", "outlet", "day"]).agg(
    Sess_count=("@timestamp", "count"),
    Sess_energy_mean=("energy", "mean"),
    Sess_temp_diff_mean=("diff", "mean"),
    Sess_duration_mean=("duration", "mean"),
    Sess_duration_total=("duration", "sum")
).reset_index()

ple_summary = ple_filtered.groupby(["@logStream", "outlet", "day"]).agg(
    PLE_count=("@timestamp", "count")
).reset_index()


- Merge into Master Timeline

In [ ]:
# Merge all three summaries
master_timeline = (
    ctd_summary.merge(sess_summary, on=["@logStream", "outlet", "day"], how="outer")
               .merge(ple_summary, on=["@logStream", "outlet", "day"], how="outer")
)


# Sort + clean
master_timeline["day"] = pd.to_datetime(master_timeline["day"], errors="coerce")
master_timeline = master_timeline.sort_values(["@logStream", "outlet", "day"])


### Feature Engineering

In [ ]:
def add_features(df):
    df = df.copy()
    # Energy/session
    df["Energy_per_session"] = np.where(
        df["Sess_count"] > 0,
        df["Sess_energy_mean"] / df["Sess_count"],
        np.nan
    )
    # Energy/duration
    df["Energy_per_duration"] = np.where(
        df["Sess_duration_mean"] > 0,
        df["Sess_energy_mean"] / df["Sess_duration_mean"],
        np.nan
    )
    # Ratios
    df["CTD_per_session"] = np.where(
        df["Sess_count"] > 0, df["CTD_count"] / df["Sess_count"], np.nan
    )
    df["PLE_per_session"] = np.where(
        df["Sess_count"] > 0, df["PLE_count"] / df["Sess_count"], np.nan
    )
    return df

outlet_timeline = add_features(master_timeline)


### Step 4 – Scoring

In [ ]:
def score_outlets_combined(outlet_timeline, rolling_window=60, min_sessions=30):
    results = []
    for (charger, outlet), df in outlet_timeline.groupby(["@logStream", "outlet"]):
        if df["Sess_count"].sum() < min_sessions:
            continue

        temp = df["Sess_temp_diff_mean"].fillna(0)
        temp_med = temp.tail(90).median()
        temp_slope = np.gradient(temp.values) if len(temp) >= 3 else [0]

        sustained_rise = (temp_slope[-90:] > 0).mean() > 0.6 if len(temp) >= 90 else False
        sharp_rise_month = temp.pct_change(30).dropna().gt(0.8).any()
        sharp_rise_week = temp.pct_change(7).dropna().gt(2.0).any()

        # --- Rule contributions ---
        contributions = {}
        temp_score = 0

        if temp_med > 5:
            contributions["Median >5°C"] = 3
            temp_score += 3
        if temp_med > 10:
            contributions["Median >10°C"] = 2
            temp_score += 2
        if sustained_rise:
            contributions["Sustained rise (90d)"] = 3
            temp_score += 3
        if sharp_rise_month:
            contributions["80% rise in 30d"] = 3
            temp_score += 3
        if sharp_rise_week:
            contributions["200% rise in 7d"] = 3
            temp_score += 3

        stability_score = 1 if df["Sess_count"].sum() > 1000 else 0
        if stability_score:
            contributions["Stable session support"] = 1

        final_score = temp_score + stability_score

        results.append({
            "@logStream": charger,
            "outlet": outlet,
            "Final_score": final_score,
            "Contributions": contributions
        })

    ranking_df = pd.DataFrame(results).sort_values("Final_score", ascending=False).reset_index(drop=True)
    ranking_df["Rank"] = ranking_df.index + 1
    return ranking_df


### Step 5 – Plotting

In [ ]:
# === Trend helper (with rise/fall markers) ===
def _add_trends(ax, x_dates, y, rolling_window=21, base_color="C0", label="Series"):
    """
    Plot raw series and its rolling mean trend on the given axis.
    Adds markers when a sustained upward or downward trend starts.
    """
    ax.plot(x_dates, y, marker='o', linestyle='-', color=base_color, alpha=0.5, label=label)

    y_roll = pd.Series(y, index=pd.to_datetime(x_dates)).rolling(
        window=rolling_window, min_periods=1
    ).mean()

    ax.plot(x_dates, y_roll.values, linestyle='--', linewidth=2,
            color="black", label=f"Trend ({rolling_window}d rolling)")

    # Skip rise/fall markers if too few points
    if len(y_roll.dropna()) < 5:
        return

    sign = np.sign(y_roll.diff().fillna(0).values)
    for i in range(len(sign) - 3):
        if all(sign[i:i+3] > 0):  # upward
            ax.annotate("↑ rise", (x_dates.iloc[i], y_roll.iloc[i]),
                        xytext=(0, 10), textcoords="offset points",
                        color=base_color, fontsize=9,
                        arrowprops=dict(arrowstyle="->", color=base_color))
            break


# === Outlier capping helper ===
def cap_outliers(series, upper_quantile=0.99):
    s = pd.to_numeric(series, errors="coerce")
    if s.dropna().empty:
        return s
    cap_value = s.quantile(upper_quantile)
    return np.minimum(s, cap_value)


# === Updated export function with score & reasons in title ===
def export_all_outlets_with_trends(outlet_timeline,
                                   ranking_df=None,
                                   output_dir="../plots_alloutlet_timelines_Grouped",
                                   rolling_window=30,
                                   duration_unit_label="min"):
    import os
    os.makedirs(output_dir, exist_ok=True)

    for (charger, outlet), df in outlet_timeline.groupby(["@logStream", "outlet"]):
        if df.empty:
            continue

        df = df.sort_values("day").copy()
        df["day"] = pd.to_datetime(df["day"], errors="coerce")

        # --- Apply outlier capping ---
        features_to_cap = {
            "Energy_per_session": 0.90,
            "Energy_per_duration": 0.90,
            "Sess_temp_diff_mean": 0.90,
            "Sess_duration_mean": 0.90,
            "CTD_count": 0.99,
            "PLE_count": 0.99,
            "CTD_per_session": 0.90,
            "PLE_per_session": 0.90,
        }
        for col, q in features_to_cap.items():
            if col in df:
                df[col] = cap_outliers(df[col], q)

        # --- Lookup score, rank, and reasons ---
        score_str, rank_str, reasons_str = "", "", ""
        if ranking_df is not None:
            row = ranking_df[(ranking_df["@logStream"] == charger) &
                             (ranking_df["outlet"] == outlet)]
            if not row.empty:
                score = row["Final_score"].values[0]
                rank = row["Rank"].values[0] if "Rank" in row.columns else row.index[0] + 1
                score_str = f" | Score: {score:.1f}"
                rank_str = f" | Rank: {rank}"
                if "Reasons" in row.columns:
                    reasons_str = f"\nReasons: {row['Reasons'].values[0]}"

        # --- Figure setup ---
        fig, axs = plt.subplots(5, 1, figsize=(18, 20), sharex=True)
        fig.suptitle(f"{charger} – Outlet {outlet}{score_str}{rank_str}{reasons_str}",
                     fontsize=14)

        # 1) Energy Usage (Wh/session + Wh/min)
        if "Energy_per_session" in df:
            _add_trends(axs[0], df["day"], df["Energy_per_session"].to_numpy(dtype=float),
                        rolling_window, base_color="green", label="Wh/Session")
        if "Energy_per_duration" in df:
            _add_trends(axs[0], df["day"], df["Energy_per_duration"].to_numpy(dtype=float),
                        rolling_window, base_color="darkgreen", label=f"Wh/{duration_unit_label}")
        axs[0].set_title("Energy Usage")
        axs[0].set_ylabel("Wh"); axs[0].legend(loc="upper left")

        # 2) Session Temperature Diff
        if "Sess_temp_diff_mean" in df:
            _add_trends(axs[1], df["day"], df["Sess_temp_diff_mean"].to_numpy(dtype=float),
                        rolling_window, base_color="orange", label="Temp Diff (°C)")
        axs[1].set_title("Session Temperature Diff (°C)")
        axs[1].set_ylabel("°C"); axs[1].legend(loc="upper left")

        # 3) Counts (CTD + PLE)
        if "CTD_count" in df:
            _add_trends(axs[2], df["day"], df["CTD_count"].to_numpy(dtype=float),
                        rolling_window, base_color="brown", label="CTD Count")
        if "PLE_count" in df:
            _add_trends(axs[2], df["day"], df["PLE_count"].to_numpy(dtype=float),
                        rolling_window, base_color="blue", label="PLE Count")
        axs[2].set_title("Event Counts")
        axs[2].set_ylabel("count"); axs[2].legend(loc="upper left")

        # 4) Ratios (CTD/session + PLE/session)
        if "CTD_per_session" in df:
            _add_trends(axs[3], df["day"], df["CTD_per_session"].to_numpy(dtype=float),
                        rolling_window, base_color="red", label="CTD per Session")
        if "PLE_per_session" in df:
            _add_trends(axs[3], df["day"], df["PLE_per_session"].to_numpy(dtype=float),
                        rolling_window, base_color="blue", label="PLE per Session")
        axs[3].axhline(1, color="gray", linestyle="--", alpha=0.6)
        axs[3].set_title("Ratios (per Session)")
        axs[3].set_ylabel("ratio"); axs[3].legend(loc="upper left")

        # 5) Session Duration
        if "Sess_duration_mean" in df:
            _add_trends(axs[4], df["day"], df["Sess_duration_mean"].to_numpy(dtype=float),
                        rolling_window, base_color="purple", label=f"Duration ({duration_unit_label})")
        axs[4].set_title(f"Session Duration (mean, {duration_unit_label})")
        axs[4].set_ylabel(duration_unit_label); axs[4].legend(loc="upper left")

        # Format x-axis
        axs[4].xaxis.set_major_locator(mdates.MonthLocator())
        axs[4].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        plt.xticks(rotation=45)

        plt.tight_layout(rect=[0, 0.02, 1, 0.97])

        # --- Save file with rank in name if available ---
        if ranking_df is not None and not row.empty:
            filename = f"Rank{rank}_{charger}_Outlet{outlet}.png".replace("/", "_")
        else:
            filename = f"{charger}_Outlet{outlet}_grouped.png".replace("/", "_")

        plt.savefig(os.path.join(output_dir, filename), dpi=150)
        plt.close(fig)

    print(f"✅ Batch export done: grouped plots saved to {output_dir}")


### Step 6 – Export SelectedOutlets_top20_211025

In [ ]:
def export_top_outlets(outlet_timeline, ranking_df, n=20, output_dir="../plots_SelectedOutlets_top20_211025"):
    import os
    os.makedirs(output_dir, exist_ok=True)

    # Take the top N from the ranking list
    top_list = ranking_df.head(n).copy()

    # Create set of charger–outlet pairs
    selected_pairs = set(zip(top_list["@logStream"], top_list["outlet"]))

    # Filter outlet_timeline to just those pairs
    filtered = outlet_timeline[outlet_timeline[["@logStream", "outlet"]]
                               .apply(tuple, axis=1)
                               .isin(selected_pairs)]

    # Reuse your export function
    export_all_outlets_with_trends(filtered,
                                   ranking_df=ranking_df,
                                   output_dir=output_dir,
                                   rolling_window=30,
                                   duration_unit_label="min")

    # --- Save the top list table ---
    table_path = os.path.join(output_dir, "SelectedOutlets_top20_table.xlsx")
    top_list.to_excel(table_path, index=False)

    print(f"✅ Exported plots + saved Top20 table to {table_path}")

    return top_list


    # 1. Build ranking
ranking_df = score_outlets_combined(outlet_timeline, rolling_window=60)

# 2. Export top 20 (plots + table)
top20 = export_top_outlets(outlet_timeline, ranking_df, n=20, output_dir="../plots_SelectedOutlets_top20_211025")

# 3. Display table inline in notebook
display(top20)



In [ ]:
# === Export top-N chargers in "csonf21|afnlo23|..." format ===
def export_top_charger_list(top_df, n=20):
    chargers = top_df.head(n)["@logStream"].unique()
    charger_str = "|".join(chargers)
    print("Pipe-separated charger list:")
    print(charger_str)
    return charger_str

# Usage (after you have top20 from export_top_outlets):
pipe_str = export_top_charger_list(top20, n=20)


In [ ]:
# === Export top-N charger-outlet pairs in "csonf21-1|csonf21-2|afnlo23-1|..." format ===
def export_top_charger_outlets(top_df, n=20):
    pairs = top_df.head(n)[["@logStream", "outlet"]].dropna()
    pairs_str = "|".join([f"{row['@logStream']}-{int(row['outlet'])}" for _, row in pairs.iterrows()])
    print("Pipe-separated charger-outlet list:")
    print(pairs_str)
    return pairs_str

# Usage (after you have top20 from export_top_outlets):
pipe_str = export_top_charger_outlets(top20, n=20)
